# Prompt 2 — Limited endpoint diagnostic
Deterministic, diagnostic-only comparison of the two historical checkpoints that actually exist. `best.pt` remains `best_legacy_unknown_epoch`; `last.pt`/`latest.pt` must contain internal epoch 129. This notebook does not train, migrate, derive, or overwrite checkpoints, Prompt 1 artifacts, or completed analysis outputs.

In [ ]:
from pathlib import Path

# Single editable configuration cell.
DRIVE_ROOT = Path('/content/drive/MyDrive/ToothFairy/ToothFairy3/iac_runs')
DATASET_ROOT = DRIVE_ROOT / 'dataset_cache_colab_v1/Dataset801_IAC_LR'
TRACKB_CACHE_ROOT = DRIVE_ROOT / 'sdf_cache_backup'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
SPLITS_PATH = DRIVE_ROOT / 'configs_cache/splits.json'
RUNS_ROOT = DRIVE_ROOT
RUN_DIR = DRIVE_ROOT / 'flow_fold0'
BEST_CHECKPOINT = RUN_DIR / 'best.pt'
LAST_CHECKPOINT_CANDIDATES = (RUN_DIR / 'last.pt', RUN_DIR / 'latest.pt')
IDENTITY_BASELINE = OUTPUT_ROOT / 'baselines/identity_prior.json'
ANALYSIS_OUTPUT = OUTPUT_ROOT / 'analysis'
WORK_DIR = OUTPUT_ROOT / 'prompt2/limited_endpoint_probe_work'
LOCAL_TEMP_PARENT = Path('/content/prompt2_limited_endpoint_staging')
REPO_URL = 'https://github.com/ColdVI/ToothFairy3-IAC-Segmentation-Flow.git'
PINNED_COMMIT = '7b6a32686fab381a0840d6f512a6ca9647f9593a'
REPO = Path('/content/ToothFairy3-IAC-Segmentation-Flow')
DEVICE = 'cuda'
EXPECTED_GPU = 'L4'
CASES = 12
PATCHES_PER_STRATUM = 2
THICKENING_CASES = 5
BATCH_SIZE = 6
BOOTSTRAP_ITERATIONS = 2000
EXPECTED_OUTPUTS = ('shortcut_probe.csv', 'shortcut_probe_summary.json',
                    'limited_endpoint_diagnostic.pdf', 'thickening_probe.csv',
                    'shortcut_probe_manifest.json')

In [ ]:
import hashlib, json, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
if not REPO.is_dir():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', '--all', '--tags'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', PINNED_COMMIT], cwd=REPO, check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert head == PINNED_COMMIT, (head, PINNED_COMMIT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO/'requirements.txt')], check=True)
import torch
assert DEVICE == 'cuda' and torch.cuda.is_available(), 'Select a Colab L4 GPU runtime'
gpu = torch.cuda.get_device_name(0)
assert EXPECTED_GPU.lower() in gpu.lower(), f'Expected L4, got {gpu}'
LAST_CHECKPOINT = next((path for path in LAST_CHECKPOINT_CANDIDATES if path.is_file()), None)
assert LAST_CHECKPOINT is not None, f'Neither last.pt nor latest.pt exists under {RUN_DIR}'
CHECKPOINTS = {'best_legacy_unknown_epoch': BEST_CHECKPOINT, 'epoch_129': LAST_CHECKPOINT}
for name, path in [('DRIVE_ROOT', DRIVE_ROOT), ('DATASET_ROOT', DATASET_ROOT),
                   ('TRACKB_CACHE_ROOT', TRACKB_CACHE_ROOT), ('SPLITS_PATH', SPLITS_PATH),
                   ('RUN_DIR', RUN_DIR), ('BEST_CHECKPOINT', BEST_CHECKPOINT),
                   ('LAST_CHECKPOINT', LAST_CHECKPOINT), ('IDENTITY_BASELINE', IDENTITY_BASELINE)]:
    assert path.exists(), f'{name} missing: {path}'
LOCAL_TEMP_PARENT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
immutable_before = {str(path): sha256(path) for path in (*CHECKPOINTS.values(), IDENTITY_BASELINE)}
print({'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': gpu, 'commit': head})

In [ ]:
readiness_cmd = [sys.executable, 'analysis/shortcut_probe.py',
    '--identity-baseline', str(IDENTITY_BASELINE), '--runs-root', str(RUNS_ROOT),
    '--run-dir', str(RUN_DIR), '--readiness-only']
for label, path in CHECKPOINTS.items():
    readiness_cmd += ['--checkpoint', f'{label}={path}']
subprocess.run(readiness_cmd, cwd=REPO, check=True)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/'], cwd=REPO, check=True)

In [ ]:
existing = [name for name in EXPECTED_OUTPUTS if (ANALYSIS_OUTPUT/name).exists()]
assert len(existing) in (0, len(EXPECTED_OUTPUTS)), f'Partial output set requires manual inspection: {existing}'
if not existing:
    probe_cmd = [sys.executable, 'analysis/shortcut_probe.py',
        '--identity-baseline', str(IDENTITY_BASELINE), '--runs-root', str(RUNS_ROOT),
        '--run-dir', str(RUN_DIR), '--splits', str(SPLITS_PATH),
        '--images', str(DATASET_ROOT/'imagesTr'), '--labels', str(DATASET_ROOT/'labelsTr'),
        '--gt-sdf', str(TRACKB_CACHE_ROOT/'gt_sdf'),
        '--coarse-sdf', str(TRACKB_CACHE_ROOT/'coarse_sdf'),
        '--output-dir', str(ANALYSIS_OUTPUT), '--work-dir', str(WORK_DIR),
        '--local-temp-parent', str(LOCAL_TEMP_PARENT), '--device', DEVICE,
        '--cases', str(CASES), '--patches-per-stratum', str(PATCHES_PER_STRATUM),
        '--thickening-cases', str(THICKENING_CASES), '--batch-size', str(BATCH_SIZE),
        '--bootstrap-iterations', str(BOOTSTRAP_ITERATIONS)]
    for label, path in CHECKPOINTS.items():
        probe_cmd += ['--checkpoint', f'{label}={path}']
    subprocess.run(probe_cmd, cwd=REPO, check=True)
else:
    print('Complete output set already exists; validation will decide whether it is reusable.')

In [ ]:
import csv
for name in EXPECTED_OUTPUTS:
    path = ANALYSIS_OUTPUT/name
    assert path.is_file() and path.stat().st_size > 0, path
manifest = json.loads((ANALYSIS_OUTPUT/'shortcut_probe_manifest.json').read_text())
summary = json.loads((ANALYSIS_OUTPUT/'shortcut_probe_summary.json').read_text())
flags = {'protocol_deviation': True, 'exact_epoch_trajectory_available': False,
         'historical_per_epoch_checkpoints_were_not_saved': True}
for payload in (summary, manifest):
    assert payload.get('diagnostic_only') is True
    assert all(payload.get(key) is value for key, value in flags.items())
for name, expected_sha in manifest['artifacts'].items():
    actual = sha256(ANALYSIS_OUTPUT/name)
    assert actual == expected_sha, (name, actual, expected_sha)
with (ANALYSIS_OUTPUT/'shortcut_probe.csv').open() as handle:
    rows = list(csv.DictReader(handle))
assert rows and {row['checkpoint_label'] for row in rows} == set(CHECKPOINTS)
assert {row['checkpoint_epoch'] for row in rows if row['checkpoint_label'] == 'best_legacy_unknown_epoch'} == {''}
assert {row['checkpoint_epoch'] for row in rows if row['checkpoint_label'] == 'epoch_129'} == {'129'}
assert {row['stratum'] for row in rows} == {'foreground', 'pure_background'}
assert {'full', 'cbct_zero', 'cbct_noise', 'cbct_shuffle', 'prior_zero', 'prior_swap'} <= {row['intervention'] for row in rows}
assert (ANALYSIS_OUTPUT/'limited_endpoint_diagnostic.pdf').read_bytes().startswith(b'%PDF')
immutable_after = {str(path): sha256(path) for path in (*CHECKPOINTS.values(), IDENTITY_BASELINE)}
assert immutable_after == immutable_before, 'An immutable checkpoint or Prompt 1 artifact changed'
print('Limited endpoint output validation PASS:', ANALYSIS_OUTPUT)

In [ ]:
print('Prompt 2 limited endpoint diagnostic complete')
print('Git:', manifest['git']['head'])
print('Checkpoints:', {label: {'epoch': item['internal_epoch'], 'sha256': item['sha256']} for label, item in manifest['checkpoints'].items()})
print('Rows:', summary['counts'])
print(summary['claim_limit'])
print('Artifacts:', *[str(ANALYSIS_OUTPUT/name) for name in EXPECTED_OUTPUTS], sep='\n- ')